# Customer Review Analyzer

## 1 — Project Overview

This notebook analyzes a collection of **customer reviews** and extracts:

- **Sentiment** (positive / neutral / negative) per review
- **Topics and themes** discussed across reviews
- **Pain points** — what's frustrating customers
- **Feature requests** — what customers want next
- **Executive summary** — a dashboard-style overview

**End-to-end pipeline:**

```
Raw reviews (text + rating)
  → Sentiment classification (rule-based + LLM)
    → Topic extraction (TF-IDF keyword clusters)
      → Pain point detection (negative + keyword patterns)
        → Feature request detection (wish/want patterns)
          → Summary dashboard
```

**Engineering patterns taught:**

| Pattern | Description |
|---------|-------------|
| Lexicon-based sentiment | VADER-style word scoring — no model needed |
| TF-IDF topic extraction | Find distinctive terms per review cluster |
| Pattern-based IE | Extract pain points and feature requests with regex |
| LLM-enhanced analysis | Richer extraction when Ollama is available |
| Dashboard in notebook | Visualize results with text-based charts |

**Stack:** Python standard library + scikit-learn. Ollama optional for LLM-powered analysis.

## 2 — Learning Goals

By the end of this notebook you will be able to:

1. **Classify sentiment** using a lexicon-based approach (no pretrained model needed)
2. **Extract topics** from a collection of reviews using TF-IDF
3. **Detect pain points** with pattern matching on negative reviews
4. **Identify feature requests** from customer language patterns
5. **Build a text-based analytics dashboard** in a notebook
6. **Compare rule-based vs. LLM-based analysis** and understand the quality tradeoff

## 3 — Problem Statement

**Problem:** A product team has hundreds of customer reviews but no time to read them all. They need:
- Overall sentiment breakdown (are customers happy?)
- What themes keep coming up? (battery, comfort, sound)
- What are the top complaints? (pain points)
- What do customers wish the product had? (feature requests)

**Why it's hard:**
- Reviews mix praise and complaints in the same sentence ("Great sound BUT terrible battery")
- Customers express the same issue differently ("dies too fast", "battery doesn't last", "always charging")
- Feature requests are implicit ("would be nice if...", "I wish it had...")
- Star ratings don't capture nuance — a 3-star review might love the product but hate the price

**Our approach:** Combine rule-based extraction (fast, deterministic) with optional LLM analysis (richer, deeper).

## 4 — Why This Project Matters

**Review analysis is one of the most common business NLP tasks:**
- Amazon, Apple, Google all analyze millions of reviews to guide product decisions
- SaaS companies mine support tickets and reviews for churn signals
- Product managers use review themes to prioritize their roadmap

**The business impact is concrete:**

| Insight | Business action |
|---------|----------------|
| "Battery complaints up 40% this month" | Investigate hardware issue |
| "Sound quality praised in 80% of reviews" | Use in marketing |
| "15 requests for USB-C charging" | Add to Q3 roadmap |
| "Negative sentiment spike after firmware update" | Roll back or patch |

This is not academic NLP — it directly drives product and revenue decisions.

## 5 — Dataset Overview

We use a **synthetic but realistic dataset** of 20 product reviews for wireless headphones.

| Property | Value |
|----------|-------|
| Reviews | 20 |
| Rating scale | 1-5 stars |
| Product | Wireless noise-cancelling headphones |
| Content | Mix of praise, complaints, and feature requests |
| Distribution | ~40% positive, ~30% mixed, ~30% negative |

Using synthetic data is intentional:
- We know the ground truth (which reviews contain complaints, requests)
- No privacy concerns
- Covers a range of realistic review patterns

## 6 — Environment Setup

In [ ]:
import urllib.request
import urllib.error
import json
import re
import math
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


def is_ollama_available(host: str = "http://localhost:11434") -> bool:
    """Check if Ollama server is running."""
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False


OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"
USE_OLLAMA = is_ollama_available(OLLAMA_HOST)

print(f"Ollama available: {USE_OLLAMA}")
if not USE_OLLAMA:
    print("  -> Using rule-based analysis (fully functional without LLM)")


## 7 — Data Loading (Review Dataset)

In [ ]:
REVIEWS = [
    {"id": 1, "rating": 5, "text": "Amazing sound quality! The noise cancellation is the best I have ever used. Completely blocks out airplane noise. Battery lasts a full 30 hours. Worth every penny."},
    {"id": 2, "rating": 4, "text": "Really good headphones overall. Sound is crisp and clear. My only complaint is the ear cushions get warm after about 2 hours of use. Wish they had cooling gel pads."},
    {"id": 3, "rating": 2, "text": "Disappointed with the battery life. Advertised 30 hours but I barely get 18. The Bluetooth keeps disconnecting from my laptop. Had to reconnect 5 times during a meeting."},
    {"id": 4, "rating": 5, "text": "Perfect for working from home. The microphone quality is excellent for calls. My colleagues say I sound crystal clear. Noise cancellation handles barking dogs perfectly."},
    {"id": 5, "rating": 1, "text": "Terrible build quality. The headband cracked after 3 months of normal use. For this price I expected much better durability. Returning these."},
    {"id": 6, "rating": 3, "text": "Sound quality is decent for the price. Not audiophile grade but good enough for podcasts and casual listening. I wish it had an equalizer app to customize the sound profile."},
    {"id": 7, "rating": 4, "text": "Love the multipoint connection feature. I can switch between my phone and laptop seamlessly. Battery life is good at around 25 hours. Wish the case was smaller for travel."},
    {"id": 8, "rating": 2, "text": "The noise cancellation creates an uncomfortable pressure feeling in my ears. I get headaches after 30 minutes. Also the touch controls are way too sensitive, constantly triggering by accident."},
    {"id": 9, "rating": 5, "text": "Best headphones I have owned. The spatial audio feature is incredible for movies. Build quality feels premium. Even the carrying case is well designed."},
    {"id": 10, "rating": 1, "text": "Do not buy. The left ear stopped working after 2 weeks. Customer support was unhelpful and kept asking me to reset. Total waste of money."},
    {"id": 11, "rating": 3, "text": "Decent headphones but nothing special. Noise cancellation is okay but not as good as Sony or Bose. Would be nice if they added LDAC codec support for higher quality streaming."},
    {"id": 12, "rating": 4, "text": "Very comfortable even for long sessions. The lightweight design does not cause neck strain. Folding mechanism makes them easy to carry. Sound could be a bit more bass-heavy though."},
    {"id": 13, "rating": 2, "text": "App is buggy and crashes constantly on Android. Cannot update firmware because the app freezes mid-update. The headphones themselves are fine but the software experience is awful."},
    {"id": 14, "rating": 5, "text": "Bought these for my daily commute and they are fantastic. Noise cancellation makes the subway silent. Quick charge gives 5 hours from just 10 minutes of charging."},
    {"id": 15, "rating": 3, "text": "OK product. Sound is average, comfort is good. I wish there was a wired option for when the battery dies. Would also love to see a find-my-headphones feature in the app."},
    {"id": 16, "rating": 1, "text": "Paint started peeling off after one month. The matte finish looked great at first but now it is chipping everywhere. Cheap materials for a premium price."},
    {"id": 17, "rating": 4, "text": "Great for gaming. Low latency mode works well and the surround sound effect is immersive. Only wish they had a boom mic attachment option for competitive gaming."},
    {"id": 18, "rating": 2, "text": "Too heavy for my taste. After an hour my head hurts. The clamping force is too tight even on the loosest setting. Sound quality does not justify the discomfort."},
    {"id": 19, "rating": 5, "text": "Third pair of these I have bought, one for each family member. Reliable, great sound, excellent battery. Customer support replaced my first pair when the hinge broke. Outstanding service."},
    {"id": 20, "rating": 3, "text": "Good but could be better. Wind noise is terrible when walking outside with ANC on. Transparency mode sounds robotic. Would love to see improved wind noise handling in a firmware update."},
]

print(f"Loaded {len(REVIEWS)} reviews")
print(f"Rating distribution: {dict(Counter(r['rating'] for r in REVIEWS))}")


## 8 — Data Inspection

In [ ]:
# Basic stats
ratings = [r["rating"] for r in REVIEWS]
word_counts = [len(r["text"].split()) for r in REVIEWS]

print(f"Total reviews: {len(REVIEWS)}")
print(f"Avg rating: {sum(ratings)/len(ratings):.1f}")
print(f"Rating distribution:")
for star in range(5, 0, -1):
    count = ratings.count(star)
    bar = "#" * (count * 3)
    print(f"  {star} star: {count:2d} {bar}")

print(f"\nAvg review length: {sum(word_counts)//len(word_counts)} words")
print(f"Shortest: {min(word_counts)} words | Longest: {max(word_counts)} words")

# Preview
print("\n--- Sample reviews ---")
for r in REVIEWS[:3]:
    print(f"  [{r['rating']} star] {r['text'][:100]}...")


## 9 — Preprocessing

We normalize review text for consistent analysis:
- Lowercase for matching
- Keep original text for display
- No stemming/lemmatization (keeps readability)

In [ ]:
def preprocess_text(text: str) -> str:
    """Normalize text for analysis while keeping readability."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


for r in REVIEWS:
    r["text_clean"] = preprocess_text(r["text"])

print("Preprocessing done.")
print(f"  Original: {REVIEWS[0]['text'][:80]}...")
print(f"  Cleaned:  {REVIEWS[0]['text_clean'][:80]}...")


## 10 — Baseline Approach (Lexicon-Based Sentiment)

We classify sentiment using a **word-score lexicon** — each word has a positive or negative weight. The sum of all word scores determines the sentiment.

This is the same principle behind VADER (Valence Aware Dictionary and sEntiment Reasoner), simplified.

**Why this works for reviews:**
- Review language is relatively straightforward ("amazing", "terrible", "broken")
- Negation handling ("not good") catches many edge cases
- No model download or training required

In [ ]:
# Sentiment lexicon (subset of common review words)
POSITIVE_WORDS = {
    "amazing": 3, "fantastic": 3, "excellent": 3, "outstanding": 3, "perfect": 3,
    "incredible": 3, "best": 2, "love": 2, "great": 2, "wonderful": 2,
    "good": 1, "nice": 1, "solid": 1, "decent": 1, "comfortable": 1,
    "reliable": 1, "crisp": 1, "clear": 1, "premium": 1, "immersive": 1,
    "seamlessly": 2, "impressed": 2, "recommend": 2, "worth": 1,
}

NEGATIVE_WORDS = {
    "terrible": -3, "awful": -3, "worst": -3, "hate": -3, "waste": -3,
    "disappointed": -2, "broke": -2, "broken": -2, "cracked": -2, "peeling": -2,
    "buggy": -2, "crashes": -2, "uncomfortable": -2, "headaches": -2, "hurts": -2,
    "bad": -1, "heavy": -1, "cheap": -1, "poor": -1, "annoying": -1,
    "unhelpful": -2, "disconnecting": -2, "sensitive": -1, "chipping": -2,
}

NEGATION_WORDS = {"not", "no", "never", "neither", "nor", "barely", "hardly", "cannot"}


def sentiment_score(text: str) -> Tuple[float, str]:
    """Compute sentiment score using lexicon. Returns (score, label)."""
    words = text.lower().split()
    score = 0.0
    negate = False

    for i, word in enumerate(words):
        clean = word.strip(".,!?;:'\"")

        if clean in NEGATION_WORDS:
            negate = True
            continue

        if clean in POSITIVE_WORDS:
            s = POSITIVE_WORDS[clean]
            score += -s if negate else s
            negate = False
        elif clean in NEGATIVE_WORDS:
            s = NEGATIVE_WORDS[clean]
            score += -s if negate else s
            negate = False
        else:
            # Reset negation after 2 words
            if negate and i > 0:
                negate = False

    if score > 1:
        label = "positive"
    elif score < -1:
        label = "negative"
    else:
        label = "neutral"

    return score, label


# Apply to all reviews
for r in REVIEWS:
    r["sentiment_score"], r["sentiment"] = sentiment_score(r["text"])

print("Sentiment analysis results:")
print(f"  Positive: {sum(1 for r in REVIEWS if r['sentiment'] == 'positive')}")
print(f"  Neutral:  {sum(1 for r in REVIEWS if r['sentiment'] == 'neutral')}")
print(f"  Negative: {sum(1 for r in REVIEWS if r['sentiment'] == 'negative')}")

print("\nSample results:")
for r in REVIEWS[:5]:
    print(f"  [{r['rating']} star] {r['sentiment']:8s} (score={r['sentiment_score']:+.0f}) {r['text'][:60]}...")


## 11 — Topic Extraction (TF-IDF Keywords)

We extract **key topics** from the review corpus using TF-IDF:

1. Vectorize all reviews with TF-IDF (unigrams + bigrams)
2. For each sentiment group, find the highest-weighted terms
3. These represent the topics that characterize each group

In [ ]:
def extract_topics(reviews: List[Dict], n_topics: int = 8) -> Dict[str, List[Tuple[str, float]]]:
    """Extract top topics per sentiment group using TF-IDF."""
    groups = defaultdict(list)
    for r in reviews:
        groups[r["sentiment"]].append(r["text_clean"])

    topics = {}
    for sentiment, texts in groups.items():
        if not texts:
            continue

        vectorizer = TfidfVectorizer(
            max_features=200,
            stop_words="english",
            ngram_range=(1, 2),
        )
        tfidf = vectorizer.fit_transform(texts)
        feature_names = vectorizer.get_feature_names_out()

        # Average TF-IDF scores across documents in this group
        avg_scores = np.asarray(tfidf.mean(axis=0)).flatten()
        top_indices = avg_scores.argsort()[::-1][:n_topics]

        topics[sentiment] = [
            (feature_names[i], round(float(avg_scores[i]), 4))
            for i in top_indices
        ]

    return topics


topics_by_sentiment = extract_topics(REVIEWS)

print("Top topics by sentiment group:")
for sentiment, topic_list in topics_by_sentiment.items():
    print(f"\n  {sentiment.upper()}:")
    for term, score in topic_list:
        bar = "#" * int(score * 100)
        print(f"    {term:25s} {score:.4f} {bar}")


## 12 — Pain Point & Feature Request Detection

### Pain Points
We find pain points by combining:
- Negative sentiment reviews
- Keyword patterns: "complaint", "issue", "problem", "broke", "disappointed"
- Structural patterns: "but...", "only complaint is...", "my issue is..."

### Feature Requests
We find feature requests using patterns customers naturally use:
- "I wish...", "would be nice if...", "would love to see..."
- "should have...", "needs a...", "hoping for..."
- "please add...", "want..." 

In [ ]:
PAIN_PATTERNS = [
    re.compile(r"(?:my )?(?:only |main |biggest )?complaint (?:is|was) (.+?)(?:\.|$)", re.I),
    re.compile(r"disappointed (?:with|by|in) (.+?)(?:\.|$)", re.I),
    re.compile(r"(?:the )?(?:problem|issue) (?:is|was) (.+?)(?:\.|$)", re.I),
    re.compile(r"(?:too|way too) (\w+ (?:for|after|even).+?)(?:\.|$)", re.I),
    re.compile(r"(?:keeps?|constantly|always) (\w+ing.+?)(?:\.|$)", re.I),
    re.compile(r"(?:stopped|broke|cracked|peeling|chipping) (.+?)(?:\.|$)", re.I),
]

WISH_PATTERNS = [
    re.compile(r"(?:I )?wish (?:it |they |there )?(?:had|was|were|have) (.+?)(?:\.|$)", re.I),
    re.compile(r"would (?:be nice|love|like) (?:if |to see |to have )?(.+?)(?:\.|$)", re.I),
    re.compile(r"(?:hoping|hope) (?:for|to see) (.+?)(?:\.|$)", re.I),
    re.compile(r"(?:should|needs? to) (?:have|add|include) (.+?)(?:\.|$)", re.I),
    re.compile(r"would also love (.+?)(?:\.|$)", re.I),
]


def extract_pain_points(reviews: List[Dict]) -> List[Dict[str, Any]]:
    """Extract pain points from reviews."""
    pain_points = []
    for r in reviews:
        text = r["text"]
        for pattern in PAIN_PATTERNS:
            match = pattern.search(text)
            if match:
                pain_points.append({
                    "review_id": r["id"],
                    "rating": r["rating"],
                    "pain_point": match.group(0).strip(),
                    "excerpt": match.group(1).strip() if match.lastindex else match.group(0).strip(),
                })
                break  # one per review
    return pain_points


def extract_feature_requests(reviews: List[Dict]) -> List[Dict[str, Any]]:
    """Extract feature requests from reviews."""
    requests = []
    for r in reviews:
        text = r["text"]
        for pattern in WISH_PATTERNS:
            match = pattern.search(text)
            if match:
                requests.append({
                    "review_id": r["id"],
                    "rating": r["rating"],
                    "request": match.group(0).strip(),
                })
                break
    return requests


pain_points = extract_pain_points(REVIEWS)
feature_requests = extract_feature_requests(REVIEWS)

print(f"Pain points found: {len(pain_points)}")
for p in pain_points:
    print(f"  [Review #{p['review_id']}, {p['rating']} star] {p['pain_point'][:80]}")

print(f"\nFeature requests found: {len(feature_requests)}")
for f in feature_requests:
    print(f"  [Review #{f['review_id']}, {f['rating']} star] {f['request'][:80]}")


## 13 — LLM-Enhanced Analysis (Optional)

When Ollama is available, we use the LLM for richer analysis:
- Nuanced sentiment (mixed, sarcastic)
- Better topic extraction
- Root cause analysis for complaints

In [ ]:
def call_ollama(prompt: str) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
    }
    req = urllib.request.Request(
        f"{OLLAMA_HOST}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return data.get("response", "").strip()


def llm_analyze_review(review_text: str) -> Dict[str, str]:
    prompt = f"""Analyze this customer review. Return a brief analysis with:
- Sentiment: positive, negative, mixed, or neutral
- Main topic: the primary subject (1-3 words)
- Pain point: if any complaint exists (or "none")
- Feature request: if any wish/request exists (or "none")

Review: {review_text}

Analysis:"""
    response = call_ollama(prompt)
    return {"raw_analysis": response}


# Run LLM analysis on a sample if available
if USE_OLLAMA:
    print("Running LLM analysis on 3 sample reviews...")
    for r in REVIEWS[:3]:
        result = llm_analyze_review(r["text"])
        print(f"\n[Review #{r['id']}, {r['rating']} star]")
        print(f"  {result['raw_analysis'][:200]}")
else:
    print("Ollama not available. Skipping LLM-enhanced analysis.")
    print("Rule-based results above are the primary analysis.")


## 14 — Summary Dashboard

A text-based analytics dashboard showing all key findings.

In [ ]:
def print_dashboard(reviews, pain_points, feature_requests, topics_by_sentiment):
    """Print a text-based analytics dashboard."""
    total = len(reviews)
    ratings = [r["rating"] for r in reviews]
    sentiments = Counter(r["sentiment"] for r in reviews)

    print("=" * 70)
    print("  CUSTOMER REVIEW ANALYTICS DASHBOARD")
    print("=" * 70)

    # Overall stats
    print(f"\n  OVERVIEW")
    print(f"  Total reviews: {total}")
    print(f"  Average rating: {sum(ratings)/len(ratings):.1f} / 5.0")
    print(f"  Median rating:  {sorted(ratings)[len(ratings)//2]}")

    # Rating distribution
    print(f"\n  RATING DISTRIBUTION")
    for star in range(5, 0, -1):
        count = ratings.count(star)
        pct = count / total * 100
        bar = "█" * int(pct / 2)
        print(f"    {star}★ : {count:2d} ({pct:4.0f}%) {bar}")

    # Sentiment breakdown
    print(f"\n  SENTIMENT BREAKDOWN")
    for label in ["positive", "neutral", "negative"]:
        count = sentiments.get(label, 0)
        pct = count / total * 100
        bar = "█" * int(pct / 2)
        emoji = {"positive": "+", "neutral": "~", "negative": "-"}[label]
        print(f"    [{emoji}] {label:8s}: {count:2d} ({pct:4.0f}%) {bar}")

    # Sentiment vs rating alignment
    print(f"\n  SENTIMENT vs RATING ALIGNMENT")
    mismatches = 0
    for r in reviews:
        expected = "positive" if r["rating"] >= 4 else ("negative" if r["rating"] <= 2 else "neutral")
        if r["sentiment"] != expected:
            mismatches += 1
    print(f"    Alignment rate: {(total - mismatches) / total:.0%} ({total - mismatches}/{total})")

    # Top topics
    print(f"\n  TOP TOPICS BY SENTIMENT")
    for sentiment, topic_list in topics_by_sentiment.items():
        terms = ", ".join(t[0] for t in topic_list[:4])
        print(f"    {sentiment.upper():8s}: {terms}")

    # Pain points
    print(f"\n  TOP PAIN POINTS ({len(pain_points)} detected)")
    for p in pain_points[:5]:
        print(f"    ! {p['excerpt'][:70]}")

    # Feature requests
    print(f"\n  FEATURE REQUESTS ({len(feature_requests)} detected)")
    for f in feature_requests[:5]:
        print(f"    > {f['request'][:70]}")

    # Recommendations
    print(f"\n  ACTIONABLE RECOMMENDATIONS")
    if any("battery" in p["pain_point"].lower() for p in pain_points):
        print("    1. INVESTIGATE battery life discrepancy (advertised vs actual)")
    if any("app" in p["pain_point"].lower() or "software" in p["pain_point"].lower() for p in pain_points):
        print("    2. FIX companion app stability issues (Android crashes)")
    if any("build" in p["pain_point"].lower() or "crack" in p["pain_point"].lower() or "peel" in p["pain_point"].lower() for p in pain_points):
        print("    3. IMPROVE build quality / materials durability")
    if any("comfort" in p["pain_point"].lower() or "heavy" in p["pain_point"].lower() or "tight" in p["pain_point"].lower() for p in pain_points):
        print("    4. REDESIGN for comfort (weight, clamping force, heat)")
    if feature_requests:
        print(f"    5. EVALUATE top feature requests for roadmap inclusion")

    print("\n" + "=" * 70)


print_dashboard(REVIEWS, pain_points, feature_requests, topics_by_sentiment)


## 15 — Evaluation and Interpretation

### How well does sentiment classification match star ratings?

In theory: 4-5 stars = positive, 1-2 stars = negative, 3 stars = neutral.
Our lexicon-based approach should roughly align with this.

In [ ]:
# Evaluate sentiment vs rating alignment
print("Detailed sentiment vs rating analysis:")
print("-" * 60)
print(f"{'ID':>3} | {'Rating':>6} | {'Sentiment':>9} | {'Score':>5} | {'Match':>5}")
print("-" * 60)

correct = 0
for r in REVIEWS:
    # Expected sentiment from rating
    if r["rating"] >= 4:
        expected = "positive"
    elif r["rating"] <= 2:
        expected = "negative"
    else:
        expected = "neutral"

    match = "YES" if r["sentiment"] == expected else "no"
    if r["sentiment"] == expected:
        correct += 1

    print(f"{r['id']:3d} | {r['rating']:6d} | {r['sentiment']:>9s} | {r['sentiment_score']:+5.0f} | {match:>5s}")

print("-" * 60)
accuracy = correct / len(REVIEWS)
print(f"\nOverall alignment: {correct}/{len(REVIEWS)} = {accuracy:.0%}")
print(f"\nNote: Mismatches are interesting — they reveal nuanced reviews")
print("(e.g., a 3-star review with strong positive language about some features)")


## 16 — Error Analysis / Limitations

**Lexicon-based sentiment limitations:**
- Fixed vocabulary — misses domain-specific positive/negative words
- Cannot detect sarcasm ("Oh great, another pair of headphones that break")
- Simple negation handling — "not the worst" is not properly analyzed
- No intensity modeling — "slightly annoyed" vs "furious" get similar scores

**Topic extraction limitations:**
- TF-IDF finds word patterns, not semantic themes
- "Battery dies fast" and "runs out of charge quickly" are different phrases for the same issue
- Bigrams help but don't capture the full relationship

**Pain point detection limitations:**
- Pattern-based — misses complaints phrased as stories
- Cannot distinguish resolved vs. ongoing issues
- Some complaint patterns overlap with neutral observations

**Feature request detection limitations:**
- Relies on specific language patterns ("wish", "would love")
- Misses implicit requests ("I bought a USB-C cable thinking it would work")

**What a production system would add:**
- Pretrained sentiment model (VADER, transformers-based)
- Named entity recognition for product feature mentions
- Clustering with sentence embeddings for theme detection
- Time-series analysis to track sentiment trends over weeks/months

## 17 — How to Improve It

### Production Improvement Ideas

1. **Sentence embeddings**: Use Sentence Transformers to cluster reviews by semantic similarity — catches different phrasings of the same issue
2. **Aspect-based sentiment**: Detect sentiment per aspect ("battery: negative, sound: positive") not per review
3. **Trend analysis**: Track sentiment over time to detect product issues after firmware updates or manufacturing changes
4. **Competitive benchmarking**: Compare review themes against competitor product reviews
5. **Alert system**: Auto-notify product team when negative sentiment spikes on a specific topic
6. **Review response generator**: Draft professional responses to negative reviews
7. **Priority scoring**: Rank pain points by frequency x severity x recency
8. **Integration**: Connect to review APIs (Amazon, Google Play, App Store) for live data

## Common Mistakes

1. **Using star ratings as ground truth for sentiment**
   A 4-star review can contain serious complaints ("great product but the app is unusable"). Analyze the text, not just the number.

2. **Treating all negative reviews equally**
   "Battery could be better" vs "product broke after 2 weeks" — one is feedback, the other is a defect. Distinguish severity.

3. **Ignoring neutral reviews**
   3-star reviews often contain the most actionable feature requests because the reviewer is balanced and thoughtful.

4. **Counting keyword frequency instead of review frequency**
   One review mentioning "battery" 10 times is 1 customer with a battery issue, not 10 battery complaints. Count per-review, not per-word.

5. **Not segmenting by rating**
   Topics in 5-star reviews represent strengths to promote. Topics in 1-star reviews represent fires to fight. Analyzing them together loses this distinction.

6. **Over-trusting LLM analysis**
   LLMs can hallucinate sentiment or fabricate pain points not in the text. Always spot-check LLM results against the original review.

## 18 — Practice Exercises

### Mini Challenge

**Challenge 1:** Add **aspect-based sentiment** — for each review, extract (aspect, sentiment) pairs. Example: "Amazing sound quality but terrible battery" → [(sound, positive), (battery, negative)]

**Challenge 2:** Build a **similarity matrix** — which reviews discuss similar issues? Use TF-IDF cosine similarity to find the top-3 most similar review pairs.

**Challenge 3:** Create a **review response generator** — given a negative review, draft a professional customer service response that acknowledges the issue and offers a solution.

### Try This Next
- Scrape real product reviews from Amazon using `requests` + `BeautifulSoup`
- Add time-based analysis: track sentiment trends across weeks
- Build a Streamlit dashboard to explore reviews interactively

## 19 — Final Takeaway / Summary

This notebook built a complete **customer review analysis pipeline**:

1. **Lexicon-based sentiment** — fast, deterministic, no model needed
2. **TF-IDF topic extraction** — find what each sentiment group talks about
3. **Pattern-based pain point detection** — extract complaints and their contexts
4. **Feature request mining** — identify what customers wish the product had
5. **Text-based analytics dashboard** — all insights in one view
6. **Optional LLM enhancement** — richer analysis when Ollama is available

**Key insight:** The best review analysis combines quantitative (ratings, sentiment scores) with qualitative (pain point excerpts, feature request quotes). Numbers tell you *how many*, but quotes tell you *why*.

**Production takeaway:** Start with rule-based extraction, measure its accuracy, then selectively add LLM analysis where rules fall short (sarcasm, implicit requests, root cause analysis).